# Entrenamiento y evaluación de modelos para predecir la variable **default** 

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path().resolve().parent.parent / "data/data-09-2025"
data_file = "cleaned_data_default.parquet"

df = pd.read_parquet(DATA_DIR / data_file)
df.head()

,plazo,vinculacion,v_cuota,v_prestamo,s_capital,s_intereses,aportes,garantias,valorgarantia,ctasahorros,...,actividadeconomica,estado_cliente,departamento,sexo,curtotalingresos,curtotalegresos,intestrato,actualizacion,default,puntaje_data
n_credito,,,,,,,,,,,,,,,,,,,,,
003-002-0125852-7,1827,8103,356849.0,15000000.0,12923538.0,123855,7741255,1,7741255,33042953.0,...,asalariados,1,antioquia,0,4597000.0,1500000.0,5.0,1,0,795.0
004-002-0068475-5,1826,1434,2650409.0,100460000.0,31911361.0,263265,4601706,1,4601706,3791115.0,...,asalariados,1,antioquia,0,4597000.0,650000.0,5.0,1,0,836.0
003-002-0122592-9,1826,573,791482.0,30000000.0,23844684.0,261477,530431,1,530431,94435.0,...,asalariados,1,antioquia,0,4400000.0,2000000.0,4.0,0,1,709.0
006-002-0023879-0,2922,1902,2860501.0,176000000.0,113842595.0,1008570,3023534,2,320385440,54841.0,...,educacion_basica_secundaria,1,antioquia,0,22020000.0,1500000.0,4.0,1,0,733.0
006-002-0026159-4,2557,1902,987637.0,50300000.0,38521256.0,317167,1023082,2,320385440,54841.0,...,educacion_basica_secundaria,1,antioquia,0,22020000.0,1500000.0,4.0,1,0,695.0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12860 entries, 003-002-0125852-7 to 003-002-0119478-4
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   plazo               12860 non-null  int64   
 1   vinculacion         12860 non-null  int64   
 2   v_cuota             12860 non-null  float64 
 3   v_prestamo          12860 non-null  float64 
 4   s_capital           12860 non-null  float64 
 5   s_intereses         12860 non-null  int64   
 6   aportes             12860 non-null  int64   
 7   garantias           12860 non-null  int64   
 8   valorgarantia       12860 non-null  int64   
 9   ctasahorros         12860 non-null  float64 
 10  edad                12860 non-null  float64 
 11  tipoasociado        12860 non-null  int64   
 12  actividadeconomica  12851 non-null  category
 13  estado_cliente      12860 non-null  int64   
 14  departamento        12859 non-null  category
 15  sexo         

In [3]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12850 entries, 003-002-0125852-7 to 003-002-0119478-4
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   plazo               12850 non-null  int64   
 1   vinculacion         12850 non-null  int64   
 2   v_cuota             12850 non-null  float64 
 3   v_prestamo          12850 non-null  float64 
 4   s_capital           12850 non-null  float64 
 5   s_intereses         12850 non-null  int64   
 6   aportes             12850 non-null  int64   
 7   garantias           12850 non-null  int64   
 8   valorgarantia       12850 non-null  int64   
 9   ctasahorros         12850 non-null  float64 
 10  edad                12850 non-null  float64 
 11  tipoasociado        12850 non-null  int64   
 12  actividadeconomica  12850 non-null  category
 13  estado_cliente      12850 non-null  int64   
 14  departamento        12850 non-null  category
 15  sexo         

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["default", "s_capital"]) # s_capital está muy correlacionada con v_prestamo
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (10280, 20)
Testing set size: (2570, 20)


In [5]:
from sklearn.metrics import classification_report, confusion_matrix


def print_resultados(model, X_test, y_test):
    print("reporte de clasificación:")
    print(classification_report(y_test, model.predict(X_test)), "\n")
    print("matriz de confusión:")
    print(confusion_matrix(y_test, model.predict(X_test)), "\n")
    feature_importances = pd.Series(model.feature_importances_, index=X_train.columns)
    feature_importances.sort_values(ascending=False, inplace=True)
    print("10 características más importantes:")
    print(feature_importances.head(10))
    return

In [6]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

model = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    verbose=0,
    random_state=1,
    )

train_x, val_x, train_y, val_y = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
)

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

model.fit(
    train_x,
    train_y,
    categorical_feature=cat_vars,
    eval_set=[(val_x, val_y)],
    eval_metric="logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=20)],
)

train_score = f1_score(y_train, model.predict(X_train))
test_score = f1_score(y_test, model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.172171
Train score: 0.89
Test score: 0.76


In [7]:
print_resultados(model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.93      0.95      2125
           1       0.71      0.83      0.76       445

    accuracy                           0.91      2570
   macro avg       0.84      0.88      0.85      2570
weighted avg       0.92      0.91      0.91      2570
 

matriz de confusión:
[[1972  153]
 [  75  370]] 

10 características más importantes:
s_intereses         432
vinculacion         356
valorgarantia       272
puntaje_data        254
curtotalingresos    241
ctasahorros         235
v_cuota             199
plazo               181
v_prestamo          177
aportes             173
dtype: int32


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from xgboost import XGBClassifier

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

train_x, val_x, train_y, val_y = train_test_split(
    X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
)

model = XGBClassifier(
    grow_policy="lossguide",
    tree_method="hist",
    early_stopping_rounds=20,
    eval_metric="auc",
    random_state=1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
)

model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

train_score = f1_score(y_train, model.predict(X_train_processed))
test_score = f1_score(y_test, model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.89
Test score: 0.77


In [9]:
print_resultados(model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.93      0.95      2125
           1       0.71      0.83      0.77       445

    accuracy                           0.91      2570
   macro avg       0.84      0.88      0.86      2570
weighted avg       0.92      0.91      0.92      2570
 

matriz de confusión:
[[1977  148]
 [  76  369]] 

10 características más importantes:
actualizacion       0.548045
estado_cliente      0.084171
tipoasociado        0.055300
garantias           0.033968
edad                0.030707
puntaje_data        0.026450
curtotalingresos    0.026213
v_prestamo          0.023720
departamento        0.020063
s_intereses         0.019754
dtype: float32


In [10]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=1, max_depth=5)
model.fit(X_train_processed, y_train)
train_score = f1_score(y_train, model.predict(X_train_processed))
test_score = f1_score(y_test, model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.69
Test score: 0.64


In [11]:
print_resultados(model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93      2125
           1       0.70      0.59      0.64       445

    accuracy                           0.88      2570
   macro avg       0.81      0.77      0.79      2570
weighted avg       0.88      0.88      0.88      2570
 

matriz de confusión:
[[2011  114]
 [ 182  263]] 

10 características más importantes:
actualizacion       0.280651
tipoasociado        0.253011
edad                0.159937
garantias           0.104421
v_prestamo          0.068416
aportes             0.062008
curtotalingresos    0.046182
s_intereses         0.017311
estado_cliente      0.004054
plazo               0.004006
dtype: float64


In [12]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    auto_class_weights='Balanced',
    verbose=0,
    )

train_x, val_x, train_y, val_y = train_test_split(
    X_train, 
    y_train, 
    test_size=0.2, 
    stratify=y_train, 
    random_state=1
    )

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

model.fit(
    train_x,
    train_y,
    cat_features=cat_vars,
    eval_set=(val_x, val_y),
    early_stopping_rounds=20,
    )

train_score = f1_score(y_train, model.predict(X_train))
test_score = f1_score(y_test, model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.81
Test score: 0.76


In [13]:
print_resultados(model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94      2125
           1       0.68      0.87      0.76       445

    accuracy                           0.91      2570
   macro avg       0.82      0.89      0.85      2570
weighted avg       0.92      0.91      0.91      2570
 

matriz de confusión:
[[1941  184]
 [  59  386]] 

10 características más importantes:
actualizacion       12.836181
vinculacion         12.610403
s_intereses         12.052804
tipoasociado        10.212950
ctasahorros          9.324600
puntaje_data         7.275255
valorgarantia        6.127920
curtotalingresos     5.091863
v_cuota              4.634942
aportes              4.533479
dtype: float64


# Sintonización de modelos

### LightGBM

In [14]:
import optuna

def objective(trial):
    param = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 500),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LGBMClassifier(
        **param,
        objective="binary",
        force_col_wise=True,
        random_state=1,
        verbose=-1,
    )

    model.fit(
        train_x,
        train_y,
        categorical_feature=categorical_features,
        eval_set=[(val_x, val_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

lgb_model = LGBMClassifier(
    **study.best_trial.params,
    objective="binary",
    force_col_wise=True,
    random_state=1,
)

categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

lgb_model.fit(
    X_train, 
    y_train,
    categorical_feature=categorical_features,
    )

lgb_params = lgb_model.get_params()

train_score = f1_score(y_train, lgb_model.predict(X_train))
test_score = f1_score(y_test, lgb_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 17:25:11,539] A new study created in memory with name: no-name-08523b48-8e11-45fc-b8bb-6a9613a259b1
[I 2026-03-04 17:25:11,734] Trial 0 finished with value: 0.7549019607843137 and parameters: {'lambda_l1': 6.01378367041976, 'lambda_l2': 0.005098673882026045, 'num_leaves': 152, 'feature_fraction': 0.4017833178831838, 'bagging_fraction': 0.45499210506231275, 'bagging_freq': 1, 'min_child_samples': 97, 'learning_rate': 0.02337024333112339, 'colsample_bytree': 0.860478707373975, 'scale_pos_weight': 5.43291343821562}. Best is trial 0 with value: 0.7549019607843137.
[I 2026-03-04 17:25:12,222] Trial 1 finished with value: 0.7942708333333334 and parameters: {'lambda_l1': 0.011528353459201288, 'lambda_l2': 2.2753359582716203e-05, 'num_leaves': 128, 'feature_fraction': 0.6298127866506036, 'bagging_fraction': 0.9072173825478788, 'bagging_freq': 7, 'min_child_samples': 27, 'learning_rate': 0.02509612503710623, 'colsample_bytree': 0.7629863882437732, 'scale_pos_weight': 7.05175178685

Best trial: 0.839 with params: {'lambda_l1': 0.3747784362449289, 'lambda_l2': 0.0053082886810172, 'num_leaves': 422, 'feature_fraction': 0.6149544953087348, 'bagging_fraction': 0.914826887986507, 'bagging_freq': 1, 'min_child_samples': 27, 'learning_rate': 0.13029969650559478, 'colsample_bytree': 0.5143984459906721, 'scale_pos_weight': 4.963983305588609}
Train score: 1.00
Test score: 0.81


In [15]:
print_resultados(lgb_model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.97      0.96      2125
           1       0.83      0.79      0.81       445

    accuracy                           0.93      2570
   macro avg       0.89      0.88      0.88      2570
weighted avg       0.93      0.93      0.93      2570
 

matriz de confusión:
[[2051   74]
 [  94  351]] 

10 características más importantes:
vinculacion         1814
curtotalingresos    1737
v_cuota             1727
s_intereses         1525
valorgarantia       1499
ctasahorros         1395
aportes             1263
puntaje_data        1222
edad                1189
v_prestamo          1052
dtype: int32


### XGBoost

In [16]:
def objective(trial):
    te = TargetEncoder()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 5e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1e2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    model = XGBClassifier(
        grow_policy="lossguide",
        tree_method="hist",
        early_stopping_rounds=20,
        eval_metric="auc",
        random_state=1,
    )

    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

xgb_model = XGBClassifier(
    **study.best_trial.params,
    grow_policy="lossguide",
    tree_method="hist",
    # early_stopping_rounds=20,
    # eval_metric="auc",
    random_state=1,
)

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

xgb_model.fit(X_train_processed, y_train)
xgb_params = xgb_model.get_params()

train_score = f1_score(y_train, xgb_model.predict(X_train_processed))
test_score = f1_score(y_test, xgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 17:25:42,220] A new study created in memory with name: no-name-566fa3b1-0c39-448b-9221-effbe52974e7
[I 2026-03-04 17:25:42,384] Trial 0 finished with value: 0.7877862595419848 and parameters: {'max_depth': 13, 'learning_rate': 0.025366932354950016, 'n_estimators': 185, 'subsample': 0.521460574635216, 'colsample_bytree': 0.8490296764155925, 'reg_alpha': 6.917378729305809, 'reg_lambda': 0.001578468767951115, 'max_delta_step': 8, 'gamma': 1.9014967536375957, 'min_child_weight': 9, 'scale_pos_weight': 4.008668718221314}. Best is trial 0 with value: 0.7877862595419848.
[I 2026-03-04 17:25:42,622] Trial 1 finished with value: 0.793939393939394 and parameters: {'max_depth': 13, 'learning_rate': 0.41721575093093977, 'n_estimators': 117, 'subsample': 0.8610723530924431, 'colsample_bytree': 0.7984589085982836, 'reg_alpha': 0.00109136974681034, 'reg_lambda': 0.0014312837140250932, 'max_delta_step': 6, 'gamma': 1.5044536947225313, 'min_child_weight': 7, 'scale_pos_weight': 6.54557386

Best trial: 0.812 with params: {'max_depth': 6, 'learning_rate': 0.38056109694683526, 'n_estimators': 72, 'subsample': 0.7166293354842597, 'colsample_bytree': 0.5959807086383736, 'reg_alpha': 53.566402872101925, 'reg_lambda': 0.0010402785724153334, 'max_delta_step': 0, 'gamma': 0.07263781234464341, 'min_child_weight': 5, 'scale_pos_weight': 4.990624273629218}
Train score: 0.77
Test score: 0.74


In [17]:
print_resultados(xgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.90      0.93      2125
           1       0.64      0.87      0.74       445

    accuracy                           0.89      2570
   macro avg       0.81      0.89      0.84      2570
weighted avg       0.91      0.89      0.90      2570
 

matriz de confusión:
[[1905  220]
 [  56  389]] 

10 características más importantes:
actualizacion       0.387294
estado_cliente      0.177394
tipoasociado        0.104879
curtotalingresos    0.067572
v_prestamo          0.040076
valorgarantia       0.039853
garantias           0.030547
puntaje_data        0.030283
edad                0.028573
s_intereses         0.013852
dtype: float32


### Random Forest

In [18]:
from sklearn.ensemble import RandomForestClassifier

def objective(trial):
    te = TargetEncoder()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = RandomForestClassifier(class_weight="balanced", random_state=1)

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
    }

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

rf_model = RandomForestClassifier(
    **study.best_trial.params,
    class_weight="balanced",
    random_state=1,
)

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

rf_model.fit(X_train_processed, y_train)
rf_params = rf_model.get_params()

train_score = f1_score(y_train, rf_model.predict(X_train_processed))
test_score = f1_score(y_test, rf_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 17:26:03,276] A new study created in memory with name: no-name-66c49b8f-e948-4e8f-8abf-631086d29c4d
[I 2026-03-04 17:26:04,412] Trial 0 finished with value: 0.7475409836065574 and parameters: {'n_estimators': 383, 'max_depth': 8, 'min_samples_split': 16, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7475409836065574.
[I 2026-03-04 17:26:05,533] Trial 1 finished with value: 0.7591973244147158 and parameters: {'n_estimators': 172, 'max_depth': 14, 'min_samples_split': 19, 'min_samples_leaf': 14, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.7591973244147158.
[I 2026-03-04 17:26:06,579] Trial 2 finished with value: 0.7504132231404959 and parameters: {'n_estimators': 90, 'max_depth': 3, 'min_samples_split': 7, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 1 with value: 0.7591973244147158.
[I 2026-03-04 17:26:07,634] Trial 3 finished with value: 0.7416107382550335 and parameters: {'n_estimators': 320, 'max_depth': 13, 'm

Best trial: 0.763 with params: {'n_estimators': 340, 'max_depth': 12, 'min_samples_split': 12, 'min_samples_leaf': 18, 'max_features': 'sqrt'}
Train score: 0.80
Test score: 0.74


In [19]:
print_resultados(rf_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.91      0.94      2125
           1       0.66      0.84      0.74       445

    accuracy                           0.90      2570
   macro avg       0.81      0.87      0.84      2570
weighted avg       0.91      0.90      0.90      2570
 

matriz de confusión:
[[1937  188]
 [  72  373]] 

10 características más importantes:
actualizacion       0.222295
tipoasociado        0.148347
garantias           0.138428
curtotalingresos    0.083709
edad                0.068512
puntaje_data        0.066369
v_prestamo          0.065549
estado_cliente      0.056049
valorgarantia       0.041239
s_intereses         0.036609
dtype: float64


### CatBoost

In [20]:
def objective(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 50, 1000),
        'depth': trial.suggest_int('depth', 3, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10, log=True),
        }

    categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = CatBoostClassifier(
        auto_class_weights='Balanced',
        verbose=0,
        )


    model.fit(
        train_x,
        train_y,
        cat_features=categorical_features,
        eval_set=(val_x, val_y),
        early_stopping_rounds=20,
        )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

cat_model = CatBoostClassifier(
    **study.best_trial.params,
    auto_class_weights='Balanced',
    verbose=0,
    )

cat_model.fit(
    X_train, 
    y_train,
    cat_features=categorical_features,
    )

cat_params = cat_model.get_params()

train_score = f1_score(y_train, cat_model.predict(X_train))
test_score = f1_score(y_test, cat_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")


[I 2026-03-04 17:27:53,260] A new study created in memory with name: no-name-3588f82e-36b9-41d7-8d20-d4f680829057
[I 2026-03-04 17:28:02,356] Trial 0 finished with value: 0.7875 and parameters: {'iterations': 455, 'depth': 5, 'learning_rate': 0.018645489058133596, 'l2_leaf_reg': 0.001447935763398298, 'min_child_samples': 44, 'subsample': 0.9447293347077059, 'colsample_bylevel': 0.8603154736614838, 'border_count': 132, 'random_strength': 6.563638771657359}. Best is trial 0 with value: 0.7875.
[I 2026-03-04 17:28:11,539] Trial 1 finished with value: 0.7875 and parameters: {'iterations': 793, 'depth': 4, 'learning_rate': 0.034947581109331716, 'l2_leaf_reg': 0.2658024440340733, 'min_child_samples': 32, 'subsample': 0.714322398740594, 'colsample_bylevel': 0.9671005758052551, 'border_count': 206, 'random_strength': 0.0010108665998186995}. Best is trial 0 with value: 0.7875.
[I 2026-03-04 17:28:20,394] Trial 2 finished with value: 0.7875 and parameters: {'iterations': 788, 'depth': 5, 'learni

Best trial: 0.787 with params: {'iterations': 455, 'depth': 5, 'learning_rate': 0.018645489058133596, 'l2_leaf_reg': 0.001447935763398298, 'min_child_samples': 44, 'subsample': 0.9447293347077059, 'colsample_bylevel': 0.8603154736614838, 'border_count': 132, 'random_strength': 6.563638771657359}
Train score: 0.73
Test score: 0.73


In [21]:
print_resultados(cat_model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.89      0.93      2125
           1       0.62      0.88      0.73       445

    accuracy                           0.89      2570
   macro avg       0.80      0.88      0.83      2570
weighted avg       0.91      0.89      0.89      2570
 

matriz de confusión:
[[1884  241]
 [  54  391]] 

10 características más importantes:
vinculacion           15.483250
actualizacion         12.197897
ctasahorros           10.890283
s_intereses            8.104672
tipoasociado           8.067994
puntaje_data           5.743695
garantias              4.971388
curtotalingresos       4.035020
valorgarantia          4.009319
actividadeconomica     3.943522
dtype: float64


# Modelos con Skrub

### LightGBM

In [22]:
def print_resultados_skrub(model, X_test, y_test):
    print("reporte de clasificación:")
    print(classification_report(y_test, model.predict(X_test)), "\n")
    print("matriz de confusión:")
    print(confusion_matrix(y_test, model.predict(X_test)), "\n")
    feature_importances = pd.Series(
        model.feature_importances_, index=X_train_processed.columns.to_list()
    )
    feature_importances.sort_values(ascending=False, inplace=True)
    print("10 características más importantes:")
    print(feature_importances.head(10))
    return

In [23]:
import warnings

from skrub import TableVectorizer

warnings.filterwarnings("ignore")


def objective(trial):
    param = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 500),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LGBMClassifier(
        **param,
        objective="binary",
        force_col_wise=True,
        random_state=1,
        verbose=-1,
    )

    model.fit(
        train_x,
        train_y,
        eval_set=[(val_x, val_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_lgb_model = LGBMClassifier(
    **study.best_trial.params,
    objective="binary",
    force_col_wise=True,
    random_state=1,
)

sk_lgb_model.fit(X_train_processed, y_train)
sk_lgb_params = sk_lgb_model.get_params()

train_score = f1_score(y_train, sk_lgb_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_lgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-05 07:34:11,494] A new study created in memory with name: no-name-5879a8c5-8ffd-45f7-9423-c410c12e0180
[I 2026-03-05 07:34:12,077] Trial 0 finished with value: 0.7934508816120907 and parameters: {'lambda_l1': 5.612569407104783, 'lambda_l2': 8.4974308409346e-05, 'num_leaves': 383, 'feature_fraction': 0.4225536853977322, 'bagging_fraction': 0.7740810180986706, 'bagging_freq': 2, 'min_child_samples': 51, 'learning_rate': 0.06538404860880868, 'colsample_bytree': 0.9054962101314972, 'scale_pos_weight': 5.578773878980596}. Best is trial 0 with value: 0.7934508816120907.
[I 2026-03-05 07:34:12,706] Trial 1 finished with value: 0.8111111111111111 and parameters: {'lambda_l1': 0.002040092302719474, 'lambda_l2': 0.022946580796916603, 'num_leaves': 432, 'feature_fraction': 0.545609230497526, 'bagging_fraction': 0.8953206628765962, 'bagging_freq': 5, 'min_child_samples': 64, 'learning_rate': 0.020532760888634472, 'colsample_bytree': 0.8625263159641319, 'scale_pos_weight': 3.684448246888

Best trial: 0.835 with params: {'lambda_l1': 9.49924554976976e-08, 'lambda_l2': 8.522143044481797e-08, 'num_leaves': 150, 'feature_fraction': 0.9466812718824115, 'bagging_fraction': 0.9228902507159217, 'bagging_freq': 1, 'min_child_samples': 45, 'learning_rate': 0.147445436736327, 'colsample_bytree': 0.5001816125205567, 'scale_pos_weight': 4.074137828781871}
Train score: 1.00
Test score: 0.79


In [24]:
print_resultados_skrub(sk_lgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      2125
           1       0.83      0.76      0.79       445

    accuracy                           0.93      2570
   macro avg       0.89      0.87      0.88      2570
weighted avg       0.93      0.93      0.93      2570
 

matriz de confusión:
[[2054   71]
 [ 105  340]] 

10 características más importantes:
s_intereses         1813
vinculacion         1445
ctasahorros         1440
curtotalingresos    1372
puntaje_data        1338
v_cuota             1198
valorgarantia       1146
aportes              990
edad                 880
v_prestamo           848
dtype: int32


### Skrub con modelo por defecto

In [25]:
from skrub import tabular_pipeline


def objective(trial):
    param = {
        "histgradientboostingclassifier__learning_rate": trial.suggest_float(
            "histgradientboostingclassifier__learning_rate", 0.01, 0.2
        ),  # noqa: E501
        "histgradientboostingclassifier__max_iter": trial.suggest_int(
            "histgradientboostingclassifier__max_iter", 100, 500
        ),  # noqa: E501
        "histgradientboostingclassifier__max_leaf_nodes": trial.suggest_int(
            "histgradientboostingclassifier__max_leaf_nodes", 20, 100
        ),  # noqa: E501
        "histgradientboostingclassifier__min_samples_leaf": trial.suggest_int(
            "histgradientboostingclassifier__min_samples_leaf", 1, 10
        ),  # noqa: E501
        "histgradientboostingclassifier__l2_regularization": trial.suggest_float(
            "histgradientboostingclassifier__l2_regularization", 1e-3, 10.0, log=True
        ),  # noqa: E501
        "histgradientboostingclassifier__early_stopping": trial.suggest_categorical(
            "histgradientboostingclassifier__early_stopping", [True, False]
        ),  # noqa: E501
    }

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = tabular_pipeline(estimator="classifier")

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_params = study.best_trial.params
sk_params = {
    k.replace("histgradientboostingclassifier__", ""): v for k, v in sk_params.items()
}  # noqa: E501
sk_model = tabular_pipeline(estimator="classifier")
sk_model["histgradientboostingclassifier"].set_params(**sk_params)
sk_model.fit(X_train, y_train)
sk_params = sk_model.get_params()

train_score = f1_score(y_train, sk_model.predict(X_train))
test_score = f1_score(y_test, sk_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-05 07:35:30,292] A new study created in memory with name: no-name-0e065dbd-ce49-4a22-82a9-f14927112a70
[I 2026-03-05 07:35:31,055] Trial 0 finished with value: 0.8096676737160121 and parameters: {'histgradientboostingclassifier__learning_rate': 0.09103283231578321, 'histgradientboostingclassifier__max_iter': 402, 'histgradientboostingclassifier__max_leaf_nodes': 50, 'histgradientboostingclassifier__min_samples_leaf': 3, 'histgradientboostingclassifier__l2_regularization': 3.4758499651286274, 'histgradientboostingclassifier__early_stopping': False}. Best is trial 0 with value: 0.8096676737160121.
[I 2026-03-05 07:35:31,827] Trial 1 finished with value: 0.8109756097560976 and parameters: {'histgradientboostingclassifier__learning_rate': 0.01140068195567946, 'histgradientboostingclassifier__max_iter': 411, 'histgradientboostingclassifier__max_leaf_nodes': 96, 'histgradientboostingclassifier__min_samples_leaf': 4, 'histgradientboostingclassifier__l2_regularization': 0.0211896052

Best trial: 0.818 with params: {'histgradientboostingclassifier__learning_rate': 0.18442510571182516, 'histgradientboostingclassifier__max_iter': 394, 'histgradientboostingclassifier__max_leaf_nodes': 44, 'histgradientboostingclassifier__min_samples_leaf': 4, 'histgradientboostingclassifier__l2_regularization': 6.513060965566763, 'histgradientboostingclassifier__early_stopping': False}
Train score: 1.00
Test score: 0.77


In [26]:
print("reporte de clasificación:")
print(classification_report(y_test, sk_model.predict(X_test)), "\n")
print("matriz de confusión:")
print(confusion_matrix(y_test, sk_model.predict(X_test)), "\n")

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.94      0.97      0.96      2125
           1       0.83      0.73      0.77       445

    accuracy                           0.93      2570
   macro avg       0.89      0.85      0.86      2570
weighted avg       0.92      0.93      0.92      2570
 

matriz de confusión:
[[2058   67]
 [ 122  323]] 



### XGBoost

In [27]:
def objective(trial):
    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 5e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1e2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    model = XGBClassifier(
        grow_policy="lossguide",
        tree_method="hist",
        early_stopping_rounds=20,
        eval_metric="auc",
        random_state=1,
    )

    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_xgb_model = XGBClassifier(
    **study.best_trial.params,
    grow_policy="lossguide",
    tree_method="hist",
    random_state=1,
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_xgb_model.fit(X_train_processed, y_train)
sk_xgb_params = xgb_model.get_params()

train_score = f1_score(y_train, sk_xgb_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_xgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-05 07:36:56,490] A new study created in memory with name: no-name-8d0ab1f5-98f1-4959-ab08-071fd4ebdf5f
[I 2026-03-05 07:36:57,133] Trial 0 finished with value: 0.7853881278538812 and parameters: {'max_depth': 10, 'learning_rate': 0.1708996152553045, 'n_estimators': 171, 'subsample': 0.8461334513951656, 'colsample_bytree': 0.9557151643820486, 'reg_alpha': 23.318138774635088, 'reg_lambda': 0.11897646679594882, 'max_delta_step': 6, 'gamma': 0.8439965276007462, 'min_child_weight': 6, 'scale_pos_weight': 6.718124874007489}. Best is trial 0 with value: 0.7853881278538812.
[I 2026-03-05 07:36:57,820] Trial 1 finished with value: 0.7878787878787878 and parameters: {'max_depth': 13, 'learning_rate': 0.36709211106961176, 'n_estimators': 328, 'subsample': 0.796530688548041, 'colsample_bytree': 0.9990087100977839, 'reg_alpha': 0.6819735803649496, 'reg_lambda': 0.07894771391516671, 'max_delta_step': 3, 'gamma': 1.5963751752503725, 'min_child_weight': 8, 'scale_pos_weight': 4.579494409963

Best trial: 0.812 with params: {'max_depth': 6, 'learning_rate': 0.010227544133408765, 'n_estimators': 139, 'subsample': 0.61115630005442, 'colsample_bytree': 0.5309856634838702, 'reg_alpha': 56.88810100902225, 'reg_lambda': 0.2059818221460834, 'max_delta_step': 5, 'gamma': 0.6810165543603361, 'min_child_weight': 6, 'scale_pos_weight': 5.268454484302617}
Train score: 0.71
Test score: 0.70


In [28]:
print_resultados_skrub(sk_xgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.86      0.92      2125
           1       0.58      0.89      0.70       445

    accuracy                           0.87      2570
   macro avg       0.78      0.88      0.81      2570
weighted avg       0.91      0.87      0.88      2570
 

matriz de confusión:
[[1836  289]
 [  48  397]] 

10 características más importantes:
actualizacion       0.463221
ctasahorros         0.091310
curtotalingresos    0.043133
tipoasociado        0.042200
s_intereses         0.039648
valorgarantia       0.033267
puntaje_data        0.027614
vinculacion         0.023030
aportes             0.018878
v_cuota             0.015922
dtype: float32


### Random Forest

In [29]:
def objective(trial):
    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = RandomForestClassifier(class_weight="balanced", random_state=1)

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
    }

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_rf_model = RandomForestClassifier(
    **study.best_trial.params,
    class_weight="balanced",
    random_state=1,
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_rf_model.fit(X_train_processed, y_train)
sk_rf_params = sk_rf_model.get_params()

train_score = f1_score(y_train, sk_rf_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_rf_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-05 07:38:07,899] A new study created in memory with name: no-name-24a5bf8c-f625-4697-ad84-1217ca93f89a
[I 2026-03-05 07:38:09,412] Trial 0 finished with value: 0.7238421955403087 and parameters: {'n_estimators': 294, 'max_depth': 15, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 0 with value: 0.7238421955403087.
[I 2026-03-05 07:38:10,763] Trial 1 finished with value: 0.7368421052631579 and parameters: {'n_estimators': 470, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 1 with value: 0.7368421052631579.
[I 2026-03-05 07:38:12,124] Trial 2 finished with value: 0.7325383304940375 and parameters: {'n_estimators': 218, 'max_depth': 4, 'min_samples_split': 16, 'min_samples_leaf': 16, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.7368421052631579.
[I 2026-03-05 07:38:13,452] Trial 3 finished with value: 0.7201365187713311 and parameters: {'n_estimators': 56, 'max_depth': 12, 'min_s

Best trial: 0.748 with params: {'n_estimators': 262, 'max_depth': 10, 'min_samples_split': 3, 'min_samples_leaf': 9, 'max_features': 'sqrt'}
Train score: 0.80
Test score: 0.73


In [30]:
print_resultados_skrub(sk_rf_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.92      0.94      2125
           1       0.67      0.81      0.73       445

    accuracy                           0.90      2570
   macro avg       0.81      0.86      0.83      2570
weighted avg       0.91      0.90      0.90      2570
 

matriz de confusión:
[[1946  179]
 [  85  360]] 

10 características más importantes:
actualizacion       0.204405
ctasahorros         0.124154
s_intereses         0.120318
curtotalingresos    0.090398
puntaje_data        0.065416
vinculacion         0.064719
valorgarantia       0.064600
tipoasociado        0.062958
aportes             0.048561
v_cuota             0.034130
dtype: float64


### CatBoost

In [31]:
def objective(trial):

    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        'iterations': trial.suggest_int('iterations', 50, 1000),
        'depth': trial.suggest_int('depth', 3, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10, log=True),
        }

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = CatBoostClassifier(
        auto_class_weights='Balanced',
        verbose=0,
        )


    model.fit(
        train_x,
        train_y,
        eval_set=(val_x, val_y),
        early_stopping_rounds=20,
        )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_cat_model = CatBoostClassifier(
    **study.best_trial.params,
    auto_class_weights='Balanced',
    verbose=0,
    )

sk_cat_model.fit(
    X_train_processed, 
    y_train,
    )

sk_cat_params = cat_model.get_params()

train_score = f1_score(y_train, sk_cat_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_cat_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")


[I 2026-03-05 07:40:31,627] A new study created in memory with name: no-name-d85efe00-baac-479c-8f20-ebbf3fd7d023
[I 2026-03-05 07:40:33,738] Trial 0 finished with value: 0.7868852459016393 and parameters: {'iterations': 966, 'depth': 4, 'learning_rate': 0.10721151645005648, 'l2_leaf_reg': 0.0012233073077273658, 'min_child_samples': 18, 'subsample': 0.8389405039382951, 'colsample_bylevel': 0.9676712197733858, 'border_count': 115, 'random_strength': 0.018658185533055625}. Best is trial 0 with value: 0.7868852459016393.
[I 2026-03-05 07:40:36,276] Trial 1 finished with value: 0.810880829015544 and parameters: {'iterations': 168, 'depth': 6, 'learning_rate': 0.07199416439646351, 'l2_leaf_reg': 1.1111195965980714, 'min_child_samples': 18, 'subsample': 0.9950808753150328, 'colsample_bylevel': 0.8354421761217876, 'border_count': 105, 'random_strength': 0.5823341528523674}. Best is trial 1 with value: 0.810880829015544.
[I 2026-03-05 07:40:39,029] Trial 2 finished with value: 0.80516129032258

Best trial: 0.818 with params: {'iterations': 166, 'depth': 4, 'learning_rate': 0.08134802672547704, 'l2_leaf_reg': 0.30353182154179437, 'min_child_samples': 6, 'subsample': 0.9937732137896247, 'colsample_bylevel': 0.902144385591746, 'border_count': 149, 'random_strength': 0.36872931081353844}
Train score: 0.78
Test score: 0.76


In [32]:
print_resultados_skrub(sk_cat_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94      2125
           1       0.67      0.88      0.76       445

    accuracy                           0.90      2570
   macro avg       0.82      0.89      0.85      2570
weighted avg       0.92      0.90      0.91      2570
 

matriz de confusión:
[[1929  196]
 [  55  390]] 

10 características más importantes:
s_intereses         15.717915
vinculacion         14.109242
actualizacion       13.577801
ctasahorros          9.595243
valorgarantia        7.382762
tipoasociado         6.440479
v_cuota              6.301172
curtotalingresos     5.761831
puntaje_data         5.608520
plazo                4.269220
dtype: float64


# Guardado de modelos y parámetos

In [34]:
import json

import joblib

MODELS_DIR = Path().resolve().parent / "models"
models = {
    "lgb": (lgb_model, lgb_params),
    "xgb": (xgb_model, xgb_params),
    "rf": (rf_model, rf_params),
    "cat":(cat_model, cat_params),
    "sk_lgb": (sk_lgb_model, sk_lgb_params),
    "sk_xgb": (sk_xgb_model, sk_xgb_params),
    "sk_rf": (sk_rf_model, sk_rf_params),
    "sk_cat": (sk_cat_model, sk_cat_params)
    }

# Guardar modelos en formato joblib y parámetros en formato JSON    

for name, (model, params) in models.items():
    joblib.dump(model, MODELS_DIR / f"{name}_model.joblib")
    with open(MODELS_DIR / f"{name}_params.json", "w") as f:
        json.dump(params, f)